#COMPSCI 546: Applied Information Retrieval - Spring 2026 ([website](https://groups.cs.umass.edu/zamani/compsci-546-applied-information-retrieval-spring-2026/))
##Assignment 3: Relevance Models (Total : 100 points)

**Description**

This is a coding assignment where you will implement the RM3 model for Query Expansion.

**Instructions**

* To start working on the assignment, you would first need to save the notebook to your local Google Drive. For this purpose, you can click on *Copy to Drive* button. You can alternatively click the *Share* button located at the top right corner and click on *Copy Link* under *Get Link* to get a link and copy this notebook to your Google Drive.  

*   For questions with descriptive answers, please replace the text in the cell which states "Enter your answer here!" with your answer. If you are using mathematical notation in your answers, please define the variables.
*   You should implement all the functions yourself and should not use a library or tool for the computation.
*   For coding questions, you can add code where it says "enter code here" and execute the cell to print the output.
* To create the final pdf submission file, execute *Runtime->RunAll* from the menu to re-execute all the cells and then generate a PDF using *File->Print->Save as PDF*. Make sure that the generated PDF contains all the codes and printed outputs before submission.


**Submission Details**

* Due data: March 30, 2026 at 11:59 PM (EDT).
* The final PDF must be uploaded on Gradescope.
* After copying this notebook to your Google Drive, please paste a link to it below. Use the same process given above to generate a link. ***You will not recieve any credit if you don't paste the link!*** Make sure we can access the file.
**LINK: *https://colab.research.google.com/drive/1jK2w4RctR8Cyl0D9KqUiJUKVt32YPHGN?usp=sharing***

**Academic Honesty**

Please follow the guidelines under the *Collaboration and Help* section of the course website.     

# Download input files and code

Please execute the cell below to download the input files.

In [1]:
import os
import zipfile

# 1. Download files
!gdown 1CLVPQ0HADjmE0cPthdZ628aIYc8yvMdq -O HW04.zip

# 2. Extract the file (Same as your previous code)
with zipfile.ZipFile('HW04.zip', 'r') as zip_file:
    zip_file.extractall('./')

# 3. Cleanup and Setup
if os.path.exists('HW04.zip'):
    os.remove('HW04.zip')

# We will use hw1 as our working directory
os.chdir('HW04')

Downloading...
From (original): https://drive.google.com/uc?id=1CLVPQ0HADjmE0cPthdZ628aIYc8yvMdq
From (redirected): https://drive.google.com/uc?id=1CLVPQ0HADjmE0cPthdZ628aIYc8yvMdq&confirm=t&uuid=ee035195-de51-460c-b2c2-effa8328f784
To: /content/HW04.zip
100% 32.5M/32.5M [00:00<00:00, 63.3MB/s]


In [2]:
#Setting the input files
queries_file = "queries_tok_clean_kstem"
col = "antique-collection.tok.clean_kstem"
query_pass = "query_pass"

# 1 : Initial Data Setup (5 points)

We use files from the [ANTIQUE](https://arxiv.org/pdf/1905.08957.pdf) dataset for this assignment. As described in the previous assignments, this is a passage retrieval dataset.

The description of the input files provided for this assignment is given below.

**Query File**

We randomly sampled a set of 5 queries from the test set of the ANTIQUE dataset. Each row of the input file contains the following information:

*query_id query_text*

The id and text information is tab separated. query_id is a unique identifier for a query and query text has been pre-processed to remove punctutation, tokenised and stemmed using the Krovetz stemmer.  

**Collection file**

Each row of the file consists of the following information:

*passage_id  passage_text*

The id and text information is tab separated. The passage text has been pre-processed to remove punctutation, tokenised and stemmed using the Krovetz stemmer (same as queries). The terms in the passage text can be accessed by splitting the text based on space.

**Query Feedback Passages**

This file consists of queries and 10 feedback passages corresponding to each query.  Each row of the file consists of the following information:

*query_id  passage_id1 passage_id2 ......passage_id10*

The entries are space separated. The first column is the query_id followed by the passage_ids.

In this section, you have to implement the following:

* Load the queries from the query file into a datastructure
* Load the query feedback passages into a datastructure.

You can use any additional datastructures than the suggested ones for your implementation.




In [3]:

'''
This function is used to load query file information into datastructure(s).
Return Variables:
queries - mapping from queryid to querytext
'''

def loadQueries(queries_file):
  queries = {}

  with open(queries_file, 'r', encoding='utf-8') as f:
    for line in f:
      line = line.strip()
      if not line:
        continue

      query_id, query_text = line.split('\t')
      queries[query_id] = query_text

  return queries


'''
This function is used to load feedback passages corresponding to the queries into a datastructure.
The file format is given below:
"query_id passage_id1 passage_id2 .....   passage_id10"
The entries are space separated.

Return Variables:
num_queries - number of queries in the file
feedback_pass - mapping from queryid to list of feedback passages
'''


def loadFeedbackPass(query_pass):
  feedback_pass = {}

  with open(query_pass, 'r', encoding='utf-8') as f:
    for line in f:
      line = line.strip()
      if not line:
        continue

      parts = line.split()   # split by space
      query_id = parts[0]
      passage_ids = parts[1:]

      feedback_pass[query_id] = passage_ids

  return feedback_pass


queries = loadQueries(queries_file)
feedback_pass = loadFeedbackPass(query_pass)


print ('Total Num of queries in the query file  : {0}'.format(len(queries)))
print ('Total Num of queries in the feedback file  : {0}'.format(len(feedback_pass)))



Total Num of queries in the query file  : 5
Total Num of queries in the feedback file  : 5



In the cell below, an inverted index with count has been created in memory. Please run the cell and use the variables for implementing the relevance feedback models.

In [4]:
'''
Store the feedback docids
'''

def storeFeedbackPass(query_pass):
    feedback_pass_ids = {}
    for line in open(query_pass):
      row = line.strip().split(' ')
      for p in row[1:]:
        if p not in feedback_pass_ids:
          feedback_pass_ids[p]=0
        feedback_pass_ids[p]+=1
    return feedback_pass_ids

feedback_pass_ids = storeFeedbackPass(query_pass)


'''
An inverted index with count information.
'''
class indexCount:
   pcount = 0
   ctf = {}
   sumdl = 0
   avgdl = 0
   doclen = {}
   index = {}
   probctf = {}
   feedback_pass_contents = {}

   def __init__(self, col):
     self.col = col


   def create_index(self):
     for line in open(self.col):
       pid,ptext = line.strip().split('\t')
       self.pcount+=1

       if pid not in self.doclen:
           self.doclen[pid]=0
       pfreq = {}
       for term in ptext.split(' '):
           self.doclen[pid]+=1

           if term not in pfreq:
              pfreq[term]=0
           pfreq[term]+=1


       for k,v in pfreq.items():
        if k not in self.index:
          self.index[k]={}

        self.index[k][pid]=v

       if pid in feedback_pass_ids:
          self.feedback_pass_contents[pid]=ptext


buildIndex = indexCount(col)
buildIndex.create_index()


'''
inverted index with count: nested dict with term and passage_id as key"
Example - {'the': {'2020338_0:11', '3174498_1:4'}}
'''
index = buildIndex.index

#total number of passages in the collection
num_passages = buildIndex.pcount

#dict with passageId as key and number of terms in the passage as value
doclen = buildIndex.doclen

#dict with passage id as key and passage text as value for feedback passages
feedback_pass_contents = buildIndex.feedback_pass_contents

print('Total number of passages in the collection :{0}'.format(num_passages))
print('Total number of feedback passages :{0}'.format(len(feedback_pass_contents)))


Total number of passages in the collection :403492
Total number of feedback passages :50


# 2 : Query Language Model (15 points)

In the cell below, calculate Maximum Likelihood Estimates for the Query Language Model, as follows:

$P_{MLE}(t|q)$ = $\frac{count(t,q)}{|q|}$

$count(t,q)$ - Number of times term $t$ appears in query $q$

$|q|$ - Number of tokens in query $q$

Calculate the estimates for all queries.


In [5]:
from collections import Counter
'''
In this function implement mle estimates for all queries.
Return Variables:
  mle_queries - mapping from queryid to mle estimates. The mle estimates for each query is a dict from each query word to its MLE probability.
'''

def calcMleQueries(queries):
  mle_queries = {}
  for query_id in queries:
    query_text = queries[query_id]
    query_words = query_text.split()
    query_word_counts = Counter(query_words)

    # per-query dictionary
    mle = {}

    for word, count in query_word_counts.items():
      mle[word] = count / len(query_words)

    # store it
    mle_queries[query_id] = mle

  return mle_queries

mle_queries = calcMleQueries(queries)
print('MLE estimates for qid 3396066 :{0}'.format(mle_queries['3396066']))


MLE estimates for qid 3396066 :{'why': 0.16666666666666666, 'do': 0.16666666666666666, 'airplane': 0.16666666666666666, 'fly': 0.16666666666666666, 'so': 0.16666666666666666, 'high': 0.16666666666666666}


# 3 : Relevance Model 1 (RM1) (30 points)

In the cell below, implement the RM1 feedback model, as follows.

$$P_{RM1}(t|\theta_{F}) \propto \sum_{p \in F} (p(t|\theta_{p}) \prod_{w \in q} p(w|\theta_{p}))$$

The passage language model is estimated using Dirichlet smoothing:

$p(t|\theta_{p}) = \frac{count(t,p) + \mu \cdot P_{MLE}(t|C)}{|p| + \mu}$ ($p(w|\theta_{p})$ is computed similarly)

$P_{MLE}(t|C) = \frac{cf(t)}{\sum_{t' \in V} cf(t')}$

$\mu = 1500$

$cf(t)$: the total number of times term $t$ appears across all passages in the collection (collection frequency)

$count(t,p)$ - Number of times term $t$ occurs in passage $p$

$|p|$ - Number of tokens in passage $p$

$F$: Feedback passages

$C$: the entire collection

For every query, this has to be computed for all unique terms present in feedback passages.

**Note:** Once you compute the weights for each term, you should normalize them by dividing by their sum. In other words, the RM1 probabilities for each query should sum to 1.

In [6]:
'''
In this function, first compute the collection language model P_MLE(t|C)
for all terms in the vocabulary using collection frequency.
Then implement RM1 using Dirichlet smoothing (mu=1500) and return
words and their probabilities.

Return Variables:
rm1 - mapping from queryid to RM1 probabilities.
The RM1 probabilities for each query is a dict from the words that
appear in the feedback passages to their RM1 probability.
'''
import collections

def calcCollectionLM(index, doclen):
  #enter your code here
  col_lm = {}
  total_term = sum(doclen.values())

  for term, postings in index.items():
    cf = sum(postings.values())
    col_lm[term] = cf / total_term

  return col_lm


'''
Implement RM1 with Dirichlet smoothing (mu=1500).
'''
def calcRM1(index, queries, doclen, feedback_pass, feedback_pass_contents, col_lm, mu=1500):
    rm1 = {}

    for qid, query_text in queries.items():
        query_words = query_text.split()
        query_passages = feedback_pass[qid]

        # collect all unique candidate terms from feedback passages
        candidate_terms = set()
        for pid in query_passages:
            ptext = feedback_pass_contents[pid]
            for term in ptext.split():
                candidate_terms.add(term)

        term_scores = {}

        # for each candidate term t
        for t in candidate_terms:
            score_t = 0.0

            # sum over feedback passages
            for pid in query_passages:
                p_len = doclen[pid]

                # P(t | p) with Dirichlet smoothing
                count_t_p = index.get(t, {}).get(pid, 0)
                p_t_given_p = (count_t_p + mu * col_lm.get(t, 0)) / (p_len + mu)

                # product over query words: prod P(w | p)
                query_likelihood = 1.0
                for w in query_words:
                    count_w_p = index.get(w, {}).get(pid, 0)
                    p_w_given_p = (count_w_p + mu * col_lm.get(w, 0)) / (p_len + mu)
                    query_likelihood *= p_w_given_p

                score_t += p_t_given_p * query_likelihood

            term_scores[t] = score_t

        # normalize so probabilities sum to 1
        total_score = sum(term_scores.values())
        if total_score > 0:
            for t in term_scores:
                term_scores[t] /= total_score

        rm1[qid] = term_scores

    return rm1

col_lm = calcCollectionLM(index, doclen)
rm1 = calcRM1(index, queries, doclen, feedback_pass,
              feedback_pass_contents, col_lm)
'''
Print out top 20 terms and corresponding probabilities
this assumes that the rm1 variable returned is a dict with queryid as key and dict consisting of term and probability values as the value.
You can alter this based on your implementation.
'''

rm1_scores = {}
rm1_final = {}
for k,v in rm1.items():
    if k not in rm1_final:
      rm1_final[k]=[]
    if k not in rm1_scores:
      rm1_scores[k]={}
    sorted_p = sorted(v.items(), key=lambda x: x[1], reverse=True)
    sorted_dict = collections.OrderedDict(sorted_p)
    for t,s in sorted_dict.items():
        rm1_final[k].append(t+":"+str(s))
        rm1_scores[k][t]=s

print('Top 20 Feedback terms and their RM1 probabilities for qid 3396066 :{0}'.format(rm1_final['3396066'][:20]))



Top 20 Feedback terms and their RM1 probabilities for qid 3396066 :['the:0.07372097888582692', 'to:0.04287778860054705', 'a:0.03683150988476214', 'and:0.036712895314596934', 'of:0.0289355240016342', 'you:0.02863651321160498', 'it:0.025797582859316073', 'is:0.025185452980259312', 'in:0.021428417112756788', 'that:0.018920851618799545', 'i:0.016880261433221216', 'are:0.013774832618079722', 'for:0.012210121001508737', 'do:0.01201707533470407', 'they:0.012002267445136912', 'have:0.011583744993117016', 'your:0.010942355683634823', 'on:0.009802016461763577', 'or:0.009708183893856939', 'not:0.009572859098492871']


# 4 : Relevance Model 3 (RM3) (30 points)

In the cell below, implement RM3 feedback model

$$P_{RM3}(t|\theta_{F}) = \alpha P_{RM1}(t|\theta_{F}) + (1-\alpha) P_{MLE}(t|q)$$

$P_{RM1}(t|\theta_{F})$ - Probability score returned by RM1 model in Question 3

$P_{MLE}(t|q)$ - Query Likelihood estimate calculated in Question 2

$\alpha = 0.5$

For every query, this has to be computed for all unique terms present in feedback passages and the query text.

In [10]:
'''
In this function implement RM1 and return words and their probabilities.
Return Variables:
rm3 - mapping from queryid to RM1 probabilities. The RM3 probabilities for each query is a dict from the words that appear in the feedback passages or the query to their RM3 probability.
'''
import collections

import collections

def calcRM3(rm1_scores, mle_queries, queries, doclen, feedback_pass, feedback_pass_contents):
  rm3 = {}
  alpha = 0.5

  for qid in queries:
    rm3[qid] = {}

    rm1_terms = set(rm1_scores[qid].keys())
    mle_terms = set(mle_queries[qid].keys())
    all_terms = rm1_terms.union(mle_terms)

    for term in all_terms:
      rm1_prob = rm1_scores[qid].get(term, 0)
      mle_prob = mle_queries[qid].get(term, 0)

      rm3[qid][term] = alpha * rm1_prob + (1 - alpha) * mle_prob

  return rm3


rm3 = calcRM3(rm1_scores,mle_queries,queries,doclen,feedback_pass,feedback_pass_contents)

'''
Print out top 20 terms and corresponding probabilities
this assumes that the rm3 variable returned is a dict with queryid as key and dict consisting of term and probability values as the value.
You can alter this based on your implementation.
'''

rm3_final = {}
for k,v in rm3.items():
    if k not in rm3_final:
      rm3_final[k]=[]
    sorted_p = sorted(v.items(), key=lambda x: x[1], reverse=True)
    sorted_dict = collections.OrderedDict(sorted_p)
    for t,s in sorted_dict.items():
        rm3_final[k].append(t+":"+str(s))

print('Top 20 Feedback terms and their RM3 probabilities for qid 3396066 :{0}'.format(rm3_final['3396066'][:20]))



Top 20 Feedback terms and their RM3 probabilities for qid 3396066 :['do:0.08934187100068536', 'so:0.08667455206591716', 'fly:0.08456184556104576', 'why:0.08430372555657922', 'airplane:0.08426079873717034', 'high:0.08409562523041808', 'the:0.03686048944291346', 'to:0.021438894300273525', 'a:0.01841575494238107', 'and:0.018356447657298467', 'of:0.0144677620008171', 'you:0.01431825660580249', 'it:0.012898791429658036', 'is:0.012592726490129656', 'in:0.010714208556378394', 'that:0.009460425809399773', 'i:0.008440130716610608', 'are:0.006887416309039861', 'for:0.006105060500754368', 'they:0.006001133722568456']


# 5 : Think (20 Points)

## 5.1 (10 Points)
When can query expansion with relevance models hurt retrieval performance? What mistakes does ChatGPT (or your choice of LLM) make when answering the question? Maybe the answer is partially correct. What points are missing? Please provide the LLM answer.   

## 5.2 (10 Points)
Write a question about query expansion for which ChatGPT (or your choice of LLM) would give a wrong answer. Please provide the question, the LLM answer, and the correct answer.

## 5.1:
When query expansion is performed using relevance models and feedback is not necessarily relevant or clean, it may have a negative impact on query expansion results. RM1 is more likely to be affected by high-frequency terms, which may end up dominating the query due to term frequency and smoothing. Also, query expansion results depend on the quality of results obtained in the initial run.

LLMs may not generalize query expansion results well and may not consider all the limitations. LLMs may not consider the probabilistic nature of RM1, common terms, and differences between RM1 and RM3. Also, they may not stress that RM3 helps to avoid all these problems by combining feedback and original query results.

## 5.2:

**Question:**
Does RM1 query expansion always improve retrieval performance compared to using the original query?

**Answer:**
Yes, RM1 generally improves retrieval performance because it incorporates additional terms from relevant documents, making the query more informative and increasing recall.

**Correction:**
No, RM1 does not necessarily improve the retrieval performance. In fact, it can degrade the performance by hurting the retrieval if the feedback passages are not actually relevant, which can lead to query drift. Also, RM1 can be biased towards high-frequency words, e.g., "the" and "and" due to the frequency of the terms and the use of the Dirichlet smoothing method, which can reduce the discriminative power of the query expansion. It is also heavily dependent on the quality of the original retrieval results, i.e., the quality of the top feedback passages, which can be low, and RM1 can actually exacerbate the problem. RM3 is generally preferred over RM1.